# Layout Planner — Qwen2.5-1.5B QLoRA (Colab)

Fine-tune **Qwen2.5-1.5B-Instruct** to turn a website brief into a **structure-only**
`LayoutPlan` JSON (Stage 1 of the two-stage CMS pipeline; Stage 2 = the API filler).

Replaces the older FLAN-T5 planner (too weak at JSON). Qwen is stronger at structured
output, Apache-2.0 licensed, and small enough to **QLoRA on a free Colab T4**.

**Anti-overfitting is built in:** eval on a held-out val split every epoch,
`load_best_model_at_end` on `eval_loss`, **early stopping**, weight decay, LoRA dropout,
and only 3 epochs. The loss-curve cell lets you *see* the train/val gap.

---
### Before you run
1. **Runtime ▸ Change runtime type ▸ T4 GPU** ▸ Save.
2. Run cells top to bottom.
3. Have `train.jsonl` and `val.jsonl` from `training/data/seq2seq/` ready to upload
   (Cell "Load data"). Each line is `{"input_text": "plan layout: <brief>", "target_text": "<LayoutPlan JSON>"}`.


## 1. Install dependencies

In [ ]:
!pip -q install -U "transformers>=4.45" "datasets>=2.20" "accelerate>=0.34" \
    "peft>=0.13" "bitsandbytes>=0.44" sentencepiece
import torch, transformers, peft
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "| gpu", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
print("transformers", transformers.__version__, "| peft", peft.__version__)

## 2. Config

In [ ]:
MODEL_ID   = "Qwen/Qwen2.5-1.5B-Instruct"
OUT_DIR    = "qwen-layout-planner"        # adapter output
MAX_LEN    = 2048                          # rich plans (11-15 blocks) fit comfortably

# LoRA
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05

# Training (anti-overfitting: modest epochs + early stop on val loss)
EPOCHS          = 3
LR              = 2e-4
TRAIN_BS        = 2
GRAD_ACCUM      = 8      # effective batch = 16
WEIGHT_DECAY    = 0.01
WARMUP_RATIO    = 0.03
EARLY_STOP_PAT  = 2      # stop if val loss doesn't improve for 2 evals

# System prompt — kept identical at TRAIN and INFERENCE time.
SYSTEM = (
    "You are a website STRUCTURE planner for OpenNext CMS. "
    "Given a brief, return ONE minified JSON LayoutPlan and nothing else. "
    "Shape: {meta:{business,style,websiteType,pageName,location,complexity},"
    "page:{theme:{primaryColor,secondaryColor,backgroundColor,textColor,fontFamily,borderRadius},"
    "blocks:[{id,type,order,styles,content}]}}. "
    "Rules: blocks[0] is a navigation bar and the last block is a footer (exactly one each); "
    "exactly one hero, placed before any feature/testimonial/pricing/content/stats section; "
    "every block.content is {} (NO copy); ids are unique kebab-case; order is 0..n-1; "
    "colours are hex and section backgrounds vary; "
    "complexity simple=4-6 blocks, standard=7-10, rich=11-15."
)

## 3. Load data
Upload `train.jsonl` and `val.jsonl` when prompted. *(Drive alternative is in the
commented block — mount your Drive and point `DATA_DIR` at the folder instead.)*

In [ ]:
from google.colab import files
print("Select train.jsonl AND val.jsonl from training/data/seq2seq/ ...")
up = files.upload()   # -> train.jsonl, val.jsonl

# --- Drive alternative (comment the two lines above, uncomment below) ---
# from google.colab import drive; drive.mount("/content/drive")
# DATA_DIR = "/content/drive/MyDrive/layout-planner"   # <-- your folder
# import shutil
# for f in ("train.jsonl","val.jsonl"): shutil.copy(f"{DATA_DIR}/{f}", f)

import json, os
def load_jsonl(p):
    with open(p, encoding="utf-8") as fh:
        return [json.loads(l) for l in fh if l.strip()]
train_rows, val_rows = load_jsonl("train.jsonl"), load_jsonl("val.jsonl")
print("train", len(train_rows), "| val", len(val_rows))
print("sample input :", train_rows[0]["input_text"][:90])
print("sample target:", train_rows[0]["target_text"][:90], "...")

## 4. Tokenize (loss on the JSON only)
Each row becomes a 3-turn chat (system ▸ user brief ▸ assistant JSON). We mask the
prompt tokens with `-100` so the model is trained **only** to produce the JSON, not to
echo the instruction.

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset
import numpy as np

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def encode(row):
    # Render the prompt to a STRING (tokenize=False), then tokenize ourselves.
    # Avoids the apply_chat_template(tokenize=True) return-type quirk that can
    # hand back a dict instead of a token list.
    prompt_str = tokenizer.apply_chat_template(
        [{"role": "system", "content": SYSTEM},
         {"role": "user",   "content": row["input_text"]}],
        tokenize=False, add_generation_prompt=True)
    prompt = tokenizer(prompt_str, add_special_tokens=False)["input_ids"]
    # target = the LayoutPlan JSON + the end-of-turn token (teaches the model to stop).
    target = tokenizer(row["target_text"], add_special_tokens=False)["input_ids"]
    target = target + [tokenizer.eos_token_id]
    input_ids = (prompt + target)[:MAX_LEN]
    labels    = ([-100] * len(prompt) + target)[:MAX_LEN]
    return {"input_ids": input_ids,
            "attention_mask": [1] * len(input_ids),
            "labels": labels}

def build_ds(rows):
    enc  = [encode(r) for r in rows]
    kept = [e for e in enc if any(t != -100 for t in e["labels"])]
    assert kept, (f"All {len(rows)} rows dropped (no supervised tokens). "
                  "Check that apply_chat_template renders content and rows are non-empty.")
    return Dataset.from_list(kept)

train_ds, val_ds = build_ds(train_rows), build_ds(val_rows)
lens = [len(e["input_ids"]) for e in train_ds]
print("kept train", len(train_ds), "| val", len(val_ds))
print("token length  max", max(lens), "| p95", int(np.percentile(lens, 95)), "| mean", int(np.mean(lens)))
print("(if max hits", MAX_LEN, "some plans are being truncated -- raise MAX_LEN)")


## 5. Load Qwen in 4-bit + attach LoRA (QLoRA)

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map="auto", torch_dtype=torch.float16,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

## 6. Train
Watch **`eval_loss`**: while it keeps falling the model is still learning; when it
turns up but `train_loss` keeps dropping, that's overfitting — early stopping halts
there and `load_best_model_at_end` restores the best checkpoint.

In [ ]:
from transformers import (TrainingArguments, Trainer,
                          DataCollatorForSeq2Seq, EarlyStoppingCallback)

collator = DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100)

args = TrainingArguments(
    output_dir=OUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=TRAIN_BS,
    per_device_eval_batch_size=TRAIN_BS,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine",
    fp16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=val_ds,
    data_collator=collator, processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOP_PAT)],
)
trainer.train()

## 7. Overfitting check — train vs val loss

In [ ]:
import matplotlib.pyplot as plt
hist = trainer.state.log_history
tr = [(h["epoch"], h["loss"])      for h in hist if "loss" in h and "epoch" in h]
ev = [(h["epoch"], h["eval_loss"]) for h in hist if "eval_loss" in h]
if tr: plt.plot(*zip(*tr), label="train loss")
if ev: plt.plot(*zip(*ev), marker="o", label="val loss")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend(); plt.title("Qwen layout planner"); plt.show()
print("best val loss:", round(min(e for _,e in ev), 4) if ev else "n/a")
print("Rule of thumb: if val loss > ~1.3x the final train loss, add more data or lower epochs/LoRA rank.")

## 8. Save adapter + smoke test

In [ ]:
trainer.save_model(OUT_DIR)          # LoRA adapter
tokenizer.save_pretrained(OUT_DIR)
print("saved adapter ->", OUT_DIR)

import json
def generate(brief):
    msgs = [{"role":"system","content":SYSTEM},
            {"role":"user","content": f"plan layout: {brief}"}]
    enc = tokenizer.apply_chat_template(msgs, add_generation_prompt=True,
                                        return_tensors="pt", return_dict=True).to(model.device)
    out = model.generate(**enc, max_new_tokens=1024, do_sample=False,
                         pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)

tests = ["Create a premium dental clinic website in Ahmedabad.",
         "Design a Pricing page for a modern SaaS analytics platform.",
         "Build a minimal photographer portfolio website."]
ok = 0
for b in tests:
    raw = generate(b)
    try:
        plan = json.loads(raw); ok += 1
        nb = len(plan["page"]["blocks"])
        print(f"OK  {nb:>2} blocks | first={plan['page']['blocks'][0]['type']:>10} "
              f"last={plan['page']['blocks'][-1]['type']} | {b[:40]}")
    except Exception as e:
        print(f"BAD json ({e}) | {b[:40]}\n   {raw[:160]}")
print(f"\nparsed {ok}/{len(tests)} as valid JSON")

## 9. Use it from Python (transformers + peft)
No Ollama, no GGUF. Serve the planner directly in Python: load base Qwen + the
LoRA adapter you just saved, then `generate()`. This is exactly what
`training/layout-planner/inference.py` does in the repo. Runs on CPU (slow) or GPU.


In [ ]:
# Load base + adapter fresh (mirrors inference.py) and sanity-check generation.
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch, json

tok2   = AutoTokenizer.from_pretrained(OUT_DIR)
base2  = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto")
infer  = PeftModel.from_pretrained(base2, OUT_DIR)
infer.eval()

def plan(brief):
    msgs = [{"role":"system","content":SYSTEM},
            {"role":"user","content": f"plan layout: {brief}"}]
    enc = tok2.apply_chat_template(msgs, add_generation_prompt=True,
                                   return_tensors="pt", return_dict=True).to(infer.device)
    out = infer.generate(**enc, max_new_tokens=1024, do_sample=False,
                         pad_token_id=tok2.pad_token_id)
    return tok2.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)

raw = plan("Create a premium dental clinic website in Ahmedabad.")
print(raw[:400])
print("parses:", bool(json.loads(raw)))   # should be True


In [ ]:
# Get the adapter into your repo. Pick ONE:

# (A) Zip + download the adapter folder, then unzip into
#     training/layout-planner/outputs/qwen-layout-planner/
import shutil
from google.colab import files
shutil.make_archive("qwen-layout-planner", "zip", OUT_DIR)
files.download("qwen-layout-planner.zip")

# (B) Or push the adapter to the HF Hub and pull it with from_pretrained later:
# from huggingface_hub import notebook_login
# notebook_login()
# model.push_to_hub("<your-username>/qwen-layout-planner")
# tokenizer.push_to_hub("<your-username>/qwen-layout-planner")
# # then locally: set LAYOUT_PLANNER_ADAPTER=<your-username>/qwen-layout-planner


## 10. Next steps (in your repo, not Colab)
1. Put the adapter folder at
   `training/layout-planner/outputs/qwen-layout-planner/`
   (option A above), **or** push it to the Hub and set the env var
   `LAYOUT_PLANNER_ADAPTER=<user>/qwen-layout-planner` (option B).
2. Run it in Python — `inference.py` loads base Qwen + this adapter with
   transformers/peft and the **same `SYSTEM` prompt**:
   ```
   python training/layout-planner/inference.py "Create a premium dental clinic website in Ahmedabad."
   ```
3. Evaluate on the held-out split:
   ```
   python training/layout-planner/scripts/evaluate_model.py --test training/data/seq2seq/test.jsonl
   ```
4. To call it from the Next.js CMS (Node), wrap `pipeline.py` in a tiny local
   Python service (FastAPI/Flask, one `/plan` endpoint) and POST to it from
   `generate-page-json/route.ts` (the documented hook); then hand the skeleton
   to the existing `v2.ts` filler.
